# 02 · Modelos y evaluación

Entrada: `data/autos_clean.csv` (sale del notebook 01).

Pregunta: ¿qué tan bien se estima el precio de un auto usado a partir de su aviso?

Plan (propuesta, diapositiva 8):

1. **Split train/test agrupado por `grupo_gemelo`**, una sola vez: un mismo auto publicado dos
   veces nunca queda de los dos lados.
2. **Modelo base:** mediana de log(precio) del modelo (si el modelo no está en train, la de la marca).
3. **Ridge y Lasso sobre log(precio)**, eligiendo hiperparámetros con **K-fold agrupado** solo en train:
   - `alpha` (regularización),
   - `umbral` de modelos raros (un modelo con menos avisos que el umbral en train pasa a "otro de la marca"),
   - con o sin la interacción **antigüedad × marca** (depreciación propia de cada marca).
4. **Evaluación en test** (se mira una sola vez, al final): RMSE, MAE y R².

Variables: antigüedad, log(km), marca, modelo, tipo de vendedor e indicios del título
(cilindrada, transmisión, combustible; el faltante es la categoría `sin_dato`).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA = Path("../data")
SEMILLA = 42

df = pd.read_csv(DATA / "autos_clean.csv")
CAT_TITULO = ["motor_titulo", "transmision_titulo", "combustible_titulo"]
df[CAT_TITULO] = df[CAT_TITULO].astype("string").fillna("sin_dato")
df.shape

(16893, 20)

## 1. Split train/test agrupado

In [2]:
split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEMILLA)
idx_tr, idx_te = next(split.split(df, groups=df["grupo_gemelo"]))
train, test = df.iloc[idx_tr].reset_index(drop=True), df.iloc[idx_te].reset_index(drop=True)

compartidos = set(train["grupo_gemelo"]) & set(test["grupo_gemelo"])
assert not compartidos, "hay gemelos de los dos lados"
print(f"train: {len(train)} · test: {len(test)} · grupos compartidos: {len(compartidos)}")

y_tr, y_te = train["log_precio"].to_numpy(), test["log_precio"].to_numpy()

train: 13507 · test: 3386 · grupos compartidos: 0


## 2. Métricas

El modelo predice log(precio). Se reporta:
- **en log:** RMSE y R² (lo que optimiza el modelo);
- **en dólares:** MAE y RMSE de `exp(predicción)`, y el **error porcentual mediano**, que es
  lo que entiende alguien que tasa un auto.

In [3]:
def metricas(y_log, pred_log, nombre):
    real, pred = np.exp(y_log), np.exp(pred_log)
    return {
        "modelo": nombre,
        "RMSE_log": mean_squared_error(y_log, pred_log) ** 0.5,
        "R2_log": r2_score(y_log, pred_log),
        "MAE_USD": mean_absolute_error(real, pred),
        "RMSE_USD": mean_squared_error(real, pred) ** 0.5,
        "error_%_mediano": np.median(np.abs(pred / real - 1)) * 100,
    }

## 3. Modelo base: mediana por modelo

Es la tasación "de sentido común": mirar cuánto se pide por ese modelo. Si la regresión no le gana,
no aporta nada.

In [4]:
med_modelo = train.groupby(["marca", "modelo"])["log_precio"].median()
med_marca = train.groupby("marca")["log_precio"].median()
med_global = np.median(y_tr)

def base(d):
    p = d.set_index(["marca", "modelo"]).index.map(med_modelo).to_numpy(dtype=float, copy=True)
    falta = np.isnan(p)
    p[falta] = d.loc[falta, "marca"].map(med_marca).fillna(med_global).to_numpy()
    return p

# Para comparar con la regresión de forma justa, el base también se evalúa por K-fold agrupado en train
cv = GroupKFold(n_splits=5)
cv_base = []
for a, b in cv.split(train, groups=train["grupo_gemelo"]):
    tr_k, va_k = train.iloc[a], train.iloc[b]
    mm = tr_k.groupby(["marca", "modelo"])["log_precio"].median()
    mb = tr_k.groupby("marca")["log_precio"].median()
    p = va_k.set_index(["marca", "modelo"]).index.map(mm).to_numpy(dtype=float, copy=True)
    f = np.isnan(p)
    p[f] = va_k.loc[f, "marca"].map(mb).fillna(tr_k["log_precio"].median()).to_numpy()
    cv_base.append(mean_squared_error(va_k["log_precio"], p) ** 0.5)
print(f"Base · RMSE_log en CV: {np.mean(cv_base):.4f} ± {np.std(cv_base):.4f}")

Base · RMSE_log en CV: 0.3776 ± 0.0117


## 4. Preparación de variables

`Preparar` hace, **con los conteos de train de cada fold** (para no filtrar información de validación):

- marca y modelo con menos de `umbral` avisos → `"<marca>|otro"` y `"otra_marca"`;
- one-hot de las categóricas (lo que aparece solo en validación/test queda en ceros);
- si `interaccion=True`, una columna `antigüedad × marca` por marca: cada marca tiene su propia
  pendiente de depreciación.

Las numéricas se estandarizan para que la regularización trate a todas las variables por igual.

In [5]:
CATS = ["marca_ag", "modelo_ag", "tipo_vendedor"] + CAT_TITULO

class Preparar(BaseEstimator, TransformerMixin):
    def __init__(self, umbral=10, interaccion=True):
        self.umbral = umbral
        self.interaccion = interaccion

    def _agrupar(self, X):
        X = X.copy()
        X["marca_ag"] = X["marca"].where(X["marca"].isin(self.marcas_), "otra_marca")
        clave = X["marca"] + "|" + X["modelo"]
        X["modelo_ag"] = clave.where(clave.isin(self.modelos_), X["marca_ag"] + "|otro")
        return X

    def fit(self, X, y=None):
        n_marca = X["marca"].value_counts()
        n_modelo = (X["marca"] + "|" + X["modelo"]).value_counts()
        self.marcas_ = set(n_marca[n_marca >= self.umbral].index)
        self.modelos_ = set(n_modelo[n_modelo >= self.umbral].index)
        Xa = self._agrupar(X)
        self.ohe_ = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(Xa[CATS])
        self.ohe_marca_ = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(Xa[["marca_ag"]])
        self.esc_ = StandardScaler().fit(self._numericas(Xa))
        return self

    def _numericas(self, Xa):
        num = Xa[["antiguedad", "log_km"]].to_numpy(dtype=float)
        if self.interaccion:
            m = self.ohe_marca_.transform(Xa[["marca_ag"]])
            num = np.hstack([num, m * Xa[["antiguedad"]].to_numpy(dtype=float)])
        return num

    def transform(self, X):
        Xa = self._agrupar(X)
        return np.hstack([self.esc_.transform(self._numericas(Xa)), self.ohe_.transform(Xa[CATS])])

COLS = ["marca", "modelo", "tipo_vendedor", "antiguedad", "log_km"] + CAT_TITULO
X_tr, X_te = train[COLS], test[COLS]
grupos_tr = train["grupo_gemelo"]

## 5. Selección de hiperparámetros (K-fold agrupado, solo train)

In [6]:
UMBRALES = [1, 2, 3, 5, 10, 20]  # 1 = sin agrupar
# Lasso es mucho más lento con alpha chico: grilla más acotada, centrada donde la 1.ª corrida
# encontró el óptimo (borde de 1e-5 y umbral 5).
grillas = {
    "Ridge": (Ridge(), np.logspace(-2, 3, 11), UMBRALES),
    "Lasso": (Lasso(max_iter=20_000), np.logspace(-6, -4, 5), [1, 2, 3, 5]),
}
busquedas = {}
for nombre, (est, alphas, umbrales) in grillas.items():
    pipe = Pipeline([("prep", Preparar()), ("reg", est)])
    gs = GridSearchCV(
        pipe,
        {"prep__umbral": umbrales, "prep__interaccion": [False, True], "reg__alpha": alphas},
        cv=GroupKFold(n_splits=5), scoring="neg_root_mean_squared_error", n_jobs=-1,
    )
    gs.fit(X_tr, y_tr, groups=grupos_tr)
    busquedas[nombre] = gs
    print(f"{nombre}: mejor {gs.best_params_} · RMSE_log CV {-gs.best_score_:.4f}")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Ridge: mejor {'prep__interaccion': False, 'prep__umbral': 2, 'reg__alpha': np.float64(1.0)} · RMSE_log CV 0.2670


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.210674e+00, tolerance: 4.045e-01
  model = cd_fast.enet_coordinate_descent(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.541799e-01, tolerance: 4.045e-01
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.879944e+00, tolerance: 4.064e-01
  model = cd_fast.enet_coordinate_descent(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.090873e+00, tolerance: 4.084e-01
  model = cd_fast.enet_coordinate_descent(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.684448e+00, tolerance: 4.049e-01
  model = cd_fast.enet_coordinate_descent(


... (15 outputs left out)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.790044e+00, tolerance: 4.049e-01
  model = cd_fast.enet_coordinate_descent(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.821882e+00, tolerance: 4.069e-01
  model = cd_fast.enet_coordinate_descent(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.391767e+00, tolerance: 4.084e-01
  model = cd_fast.enet_coordinate_descent(


Lasso: mejor {'prep__interaccion': False, 'prep__umbral': 2, 'reg__alpha': np.float64(3.1622776601683795e-05)} · RMSE_log CV 0.2684


### ¿Qué aporta cada decisión?

Mejor RMSE de validación para cada combinación de umbral e interacción (mínimo sobre `alpha`).

In [7]:
tablas = []
for nombre, gs in busquedas.items():
    r = pd.DataFrame(gs.cv_results_)
    t = (r.assign(RMSE_log=-r["mean_test_score"])
          .groupby(["param_prep__interaccion", "param_prep__umbral"])["RMSE_log"].min()
          .unstack().round(4))
    t.index = [f"{nombre} · interacción={i}" for i in t.index]
    tablas.append(t)
tabla_cv = pd.concat(tablas)
tabla_cv.columns = [f"umbral {u}" for u in tabla_cv.columns]
tabla_cv

,umbral 1,umbral 2,umbral 3,umbral 5,umbral 10,umbral 20
Ridge · interacción=False,0.2672,0.2670,0.2699,0.2763,0.2839,0.3004
Ridge · interacción=True,0.2722,0.2699,0.2693,0.2752,0.2806,0.2966
Lasso · interacción=False,0.2746,0.2684,0.2706,0.2767,NaN,NaN
Lasso · interacción=True,0.2790,0.2711,0.2696,0.2748,NaN,NaN


## 6. Evaluación final en test

Se reentrena cada modelo con **todo train** y los hiperparámetros elegidos (`GridSearchCV` ya lo hace)
y se mide **una sola vez** en test.

In [8]:
res = [metricas(y_te, base(test), "Base: mediana por modelo")]
for nombre, gs in busquedas.items():
    res.append(metricas(y_te, gs.predict(X_te), nombre))
resultados = pd.DataFrame(res).set_index("modelo").round(
    {"RMSE_log": 4, "R2_log": 3, "MAE_USD": 0, "RMSE_USD": 0, "error_%_mediano": 1})
resultados.to_csv(DATA / "resultados_test.csv")
resultados

,RMSE_log,R2_log,MAE_USD,RMSE_USD,error_%_mediano
modelo,,,,,
Base: mediana por modelo,0.3711,0.636,4231.0,8465.0,16.3
Ridge,0.2595,0.822,2747.0,6254.0,10.3
Lasso,0.2600,0.822,2773.0,6267.0,10.6


## 7. Guardar predicciones de test

Para el notebook 03 (residuos y depreciación): residuo = log(precio publicado) − log(precio predicho).

In [9]:
mejor = min(busquedas, key=lambda n: -busquedas[n].best_score_)
pred = busquedas[mejor].predict(X_te)
salida = test[["item_id", "marca", "modelo", "anio", "km", "precio_usd", "tipo_vendedor"]].assign(
    pred_usd=np.exp(pred).round(0), residuo=y_te - pred)
salida.to_csv(DATA / "predicciones_test.csv", index=False)
print(f"Modelo guardado para residuos: {mejor} {busquedas[mejor].best_params_}")

Modelo guardado para residuos: Ridge {'prep__interaccion': False, 'prep__umbral': 2, 'reg__alpha': np.float64(1.0)}
